# 23 — Transfer with Portmap kept, matched evaluation, prior-only baseline, labelled-sample composition

Repeats the cross-dataset part of notebook 21 with three changes and two additions.

Design log
* Label mapping: every CIC-DDoS2019 attack label maps to the DDoS family, as in the preprint's regime-2
  table (`results/demoB_transfer_regime2_rf.csv`, 1,171,936 flows). `driftbench.harmonise` has no rule
  for Portmap, so notebook 21 left Day 2's 186,960 Portmap flows out; here they are kept.
* Matched evaluation: besides the notebook-21 layout (calibrator fitted on the first p% of a day, scored
  on the remaining rows), every calibrator and the unchanged detector are also scored on one common
  evaluation part per day, the rows after the first 10%, so budgets and methods are compared on the
  same flows.
* Prior-only baseline: every evaluation flow gets the attack share of the labelled sample as its
  probability. It is the constant predictor of the paper's ECE identity, scored with the same panel.
* Labelled-sample composition: size, benign count, attack types and the detector's AUROC within each
  labelled sample, so "would a ranking check on the sample have passed" is measured, not inferred.
* Platt slope: the fitted slope of each Platt curve is recorded (sign-corrected to the paper's
  convention, positive = order kept).
* The nine source models are the same fits as notebook 21 (same split, same seed, same settings); their
  per-flow scores are stored under `results/conference/scores/` (parquet, not committed) so every
  evaluation stage reads stored scores.
* Compute: CPU. 9 model fits (3 models x 3 seeds) on 342,771 rows, 18 scoring passes over 3.27M flows,
  108 calibrator fits on at most 117,194 rows. No model API calls. Stage 2 skips models whose scores exist.

In [ ]:
# ---- Setup -------------------------------------------------------------------
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    pass

BASE = os.environ.get('DRIFT_BASE', '/content/drive/MyDrive/drift-conference')
DATA = f'{BASE}/data'
OUT  = f'{BASE}/results/conference'
SC   = f'{OUT}/scores'
PAP  = f'{OUT}/paper'
os.makedirs(SC, exist_ok=True); os.makedirs(PAP, exist_ok=True)
os.environ['DRIFT_DATA_ROOT'] = DATA
sys.path.insert(0, f'{BASE}/src')

import time, gc, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, matthews_corrcoef, roc_auc_score, average_precision_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.base import BaseEstimator, ClassifierMixin
from lightgbm import LGBMClassifier
import sklearn, lightgbm

from driftbench.splits import random_split
from driftbench.metrics import expected_calibration_error
from driftbench.drift import psi, ks_stat
from driftbench.models import time_ordered_buffer

CFG = dict(seeds=[0, 1, 2], models=['rf', 'lgbm', 'mlp'],
           fracs=[0.01, 0.05, 0.10], methods=['sigmoid', 'isotonic'], common_frac=0.10,
           source_train_frac=0.8, drift_sample=300_000, native_min_rows=50,
           ece_bins=15, dpi=600)
MODEL_NAME = {'rf': 'Random Forest', 'lgbm': 'LightGBM', 'mlp': 'MLP'}
METHOD_NAME = {'none': 'No recalibration', 'sigmoid': 'Platt', 'isotonic': 'Isotonic', 'prior': 'Prior only'}
DAY_NAME = {'regime_1': 'Day 1', 'regime_2': 'Day 2'}
print('sklearn', sklearn.__version__, '| lightgbm', lightgbm.__version__, '| pandas', pd.__version__)
print(json.dumps({k: str(v) for k, v in CFG.items()}, indent=1))

## Stage 1 — Load the cross-dataset views

Same files and the same column-name recovery as notebook 21. The target label mapping is benign versus
DDoS for every attack label, Portmap included. Prints each day's composition and where each attack
type sits in the day's time order.

In [ ]:
P_IDS = f'{DATA}/interim/CSE-CIC-IDS2018'
P_R1, P_R2 = f'{DATA}/interim/CIC-DDoS2019', f'{DATA}/interim/CIC-DDoS2019-test'
need = {
    'IDS2018 core features': f'{DATA}/processed/CSE-CIC-IDS2018/features_core.parquet',
    'IDS2018 family labels': f'{DATA}/processed/CSE-CIC-IDS2018/labels_family.parquet',
    'DDoS2019 day 1 core features (processed)': f'{DATA}/processed/CIC-DDoS2019/features_core.parquet',
    'DDoS2019 day 1 features': f'{P_R1}/features.parquet', 'DDoS2019 day 1 labels': f'{P_R1}/labels.parquet',
    'DDoS2019 day 1 metadata': f'{P_R1}/metadata.parquet',
    'DDoS2019 day 2 features': f'{P_R2}/features.parquet', 'DDoS2019 day 2 labels': f'{P_R2}/labels.parquet',
    'DDoS2019 day 2 metadata': f'{P_R2}/metadata.parquet',
}
missing = [f'{k}: {p}' for k, p in need.items() if not os.path.exists(p)]
if missing:
    raise FileNotFoundError('Missing data files:\n' + '\n'.join(missing))

def time_col(meta):
    for c in meta.columns:
        if 'timestamp' in c or 'date' in c:
            return c
    raise KeyError(f'no time column in {list(meta.columns)}')

def parse_time(series, fmt, dayfirst):
    s = series.astype(str).str.strip()
    t = pd.to_datetime(s, format=fmt, errors='coerce')
    bad = t.isna()
    if bad.any():   # second pass for rows written in another layout (e.g. without seconds)
        t[bad] = pd.to_datetime(s[bad], format='mixed', dayfirst=dayfirst, errors='coerce')
    return t

def is_benign(arr):
    return pd.Series(arr).astype(str).str.lower().str.contains('benign|normal').to_numpy()

# --- Cross-dataset views (common-core features, family labels) ---
XS_df = pd.read_parquet(need['IDS2018 core features'])
yS = pd.read_parquet(need['IDS2018 family labels'])['family'].astype(str).to_numpy()
CORE = list(XS_df.columns)
P1 = pd.read_parquet(need['DDoS2019 day 1 core features (processed)'])
assert list(P1.columns) == CORE, 'processed DDoS2019 core columns differ from IDS2018 core columns'
# The processed files carry the 2018 short names (e.g. fwd_pkt_len_max). The cleaned CIC-DDoS2019 files
# use the long names (e.g. fwd_packet_length_max). Recover the exact column mapping from regime 1, where
# the processed file and the cleaned file hold the same rows: each processed column is matched to the
# cleaned column with identical values. The mapping is then checked on every row and applied to regime 2.
import difflib
F1 = pd.read_parquet(need['DDoS2019 day 1 features'])
assert len(F1) == len(P1), 'regime 1 processed and cleaned files differ in length'
n_chk = min(200_000, len(P1))
Pv = P1.iloc[:n_chk].to_numpy(np.float64); Fv = F1.iloc[:n_chk].to_numpy(np.float64); fcols = list(F1.columns)
COLMAP, ambiguous = {}, {}
for j, c in enumerate(CORE):
    eq = [fcols[k] for k in range(Fv.shape[1])
          if np.allclose(Fv[:, k], Pv[:, j], rtol=1e-6, atol=0, equal_nan=True)]
    if not eq:
        raise ValueError(f'no cleaned CIC-DDoS2019 column matches processed column {c}')
    COLMAP[c] = c if c in eq else max(eq, key=lambda x: difflib.SequenceMatcher(None, x, c).ratio())
    if len(eq) > 1:
        ambiguous[c] = eq
full_ok = np.allclose(F1[[COLMAP[c] for c in CORE]].to_numpy(np.float32), P1.to_numpy(np.float32), equal_nan=True)
print(f'column mapping recovered for {len(COLMAP)} core features; holds on every regime 1 row: {full_ok}')
assert full_ok
renamed = {c: m for c, m in COLMAP.items() if c != m}
print('renamed:', ', '.join(f'{c} <- {m}' for c, m in renamed.items()))
del F1, Pv, Fv; _ = gc.collect()

day = {}
for r, p in [('regime_1', P_R1), ('regime_2', P_R2)]:
    f = pd.read_parquet(f'{p}/features.parquet')
    miss = [COLMAP[c] for c in CORE if COLMAP[c] not in f.columns]
    assert not miss, f'{r}: cleaned features missing {miss}'
    for c, cands in ambiguous.items():   # columns that were identical in regime 1: are they still identical here?
        present = [x for x in cands if x in f.columns]
        same = all(np.allclose(f[present[0]].to_numpy(np.float64), f[x].to_numpy(np.float64), equal_nan=True) for x in present[1:])
        if r == 'regime_2' or not same:
            print(f'{r}: {c} matched identical columns {present}; still identical here: {same}; using {COLMAP[c]}')
    f = f[[COLMAP[c] for c in CORE]].set_axis(CORE, axis=1)
    lab = pd.read_parquet(f'{p}/labels.parquet')['label'].astype(str).str.strip().to_numpy()
    met = pd.read_parquet(f'{p}/metadata.parquet')
    assert len(f) == len(lab) == len(met), f'{r} files are not row-aligned'
    t = parse_time(met[time_col(met)], '%Y-%m-%d %H:%M:%S.%f', dayfirst=False)
    day[r] = dict(X=f[CORE].to_numpy(np.float32), native=lab, t=t.to_numpy(),
                  days=sorted(t.dt.date.astype(str).unique()), unparsed=int(t.isna().sum()))
    del f
del P1
FAMS = ['benign', 'ddos']
print('common-core features:', len(CORE), '| label families:', FAMS)
assert any(pd.Series(day['regime_2']['native']).str.lower().str.contains('portmap')), 'regime_2 has no Portmap'

XT_parts, yT_parts, nT_parts, tT_parts, REG_ROWS, off = [], [], [], [], {}, 0
for r in ['regime_1', 'regime_2']:
    nat = day[r]['native']
    fam = np.where(is_benign(nat), 'benign', 'ddos')          # every CIC-DDoS2019 attack label is DDoS
    XT_parts.append(day[r]['X']); yT_parts.append(fam); nT_parts.append(nat); tT_parts.append(day[r]['t'])
    REG_ROWS[r] = np.arange(off, off + len(nat)); off += len(nat)
    vc = pd.Series(fam).value_counts()
    print(f"{r}: capture day {day[r]['days']}, unparsed timestamps {day[r]['unparsed']}, {len(nat):,} rows | " +
          ', '.join(f'{k} {v:,}' for k, v in vc.items()))
XT, yT, nT, tT = np.vstack(XT_parts), np.concatenate(yT_parts), np.concatenate(nT_parts), np.concatenate(tT_parts)
keepS = np.isin(yS, FAMS)
XS = XS_df.to_numpy(np.float32)[keepS]; yS = yS[keepS]
print(f'source (IDS2018, shared families): {len(yS):,} rows |', pd.Series(yS).value_counts().to_dict())
ORD = {r: rows[np.argsort(tT[rows], kind='stable')] for r, rows in REG_ROWS.items()}
for r, order in ORD.items():
    nat = nT[order]; n = len(order)
    print(f'\n{r}: position of each attack type in time order (share of the day at first and last flow)')
    for t in pd.Series(nat).value_counts().index:
        pos = np.where(nat == t)[0]
        print(f'  {t:16s} {len(pos):>9,} flows   from {pos[0] / n:6.1%} to {pos[-1] / n:6.1%} of the day')
del XS_df, XT_parts, day; _ = gc.collect()

## Helpers

Same model settings, label encoding, calibration wrapper and metric panel as notebook 21 (the panel was
checked against `driftbench.metrics.compute_metrics` there). `calibrate` also returns the fitted
calibrator so the Platt slope can be read.

In [ ]:
def make_model(name, seed):
    if name == 'rf':
        return RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=seed,
                                      class_weight='balanced_subsample')
    if name == 'lgbm':
        return LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63,
                              class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1)
    if name == 'mlp':
        return EncodedLabels(Pipeline([('scaler', StandardScaler()),
                                       ('mlp', MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=200,
                                                             early_stopping=True, validation_fraction=0.1,
                                                             n_iter_no_change=10, random_state=seed))]))
    raise ValueError(name)

class EncodedLabels(ClassifierMixin, BaseEstimator):
    # Fits the wrapped model on integer class codes and reports the original class names:
    # in scikit-learn 1.6 and 1.7, MLP early stopping fails on string labels.
    def __init__(self, model=None):
        self.model = model
    def fit(self, X, y):
        self.classes_, codes = np.unique(np.asarray(y).astype(str), return_inverse=True)
        self.model.fit(X, codes)
        return self
    def predict_proba(self, X):
        return self.model.predict_proba(X)
    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(1)]

class StoredProba(ClassifierMixin, BaseEstimator):
    # Passes stored probabilities through, so CalibratedClassifierCV can calibrate them.
    def __init__(self, classes=None):
        self.classes = classes
    def fit(self, X=None, y=None):
        self.classes_ = np.asarray(self.classes); return self
    def predict_proba(self, X):
        return np.asarray(X, float)
    def predict(self, X):
        return self.classes_[np.asarray(X).argmax(1)]

def calibrate(P_buf, y_buf, P_eval, classes, method):
    try:
        from sklearn.frozen import FrozenEstimator
        cal = CalibratedClassifierCV(FrozenEstimator(StoredProba(classes).fit()), method=method)
    except ImportError:
        cal = CalibratedClassifierCV(StoredProba(classes).fit(), method=method, cv='prefit')
    cal.fit(P_buf, y_buf)
    assert list(cal.classes_) == list(classes)
    return cal.predict_proba(P_eval), cal

def platt_slope(cal):
    # scikit-learn's sigmoid is expit(-(a_ * s + b_)); the paper's slope a = -a_ (positive keeps the order).
    try:
        c = cal.calibrated_classifiers_[0].calibrators[0]
        return float(-c.a_)
    except Exception:
        return np.nan

def prior_only(P_buf, y_buf, n_eval, classes):
    # Constant predictor: every evaluation flow gets the attack share of the labelled sample.
    rho = float((~is_benign(y_buf)).mean())
    ben_col = np.where(is_benign(classes))[0][0]
    P = np.empty((n_eval, len(classes)), float); P[:, ben_col] = 1 - rho; P[:, 1 - ben_col] = rho
    return P, rho

def two_col_scores(s_att, classes):
    # Stored P(attack) back to the two-column panel layout.
    ben_col = np.where(is_benign(classes))[0][0]
    P = np.empty((len(s_att), 2), float); P[:, ben_col] = 1 - s_att; P[:, 1 - ben_col] = s_att
    return P

def panel(y_true, P, classes):
    # Same definitions as driftbench.metrics.compute_metrics, plus attack-vs-benign scores.
    y_true = np.asarray(y_true).astype(str); classes = np.asarray(classes).astype(str)
    pred = classes[P.argmax(1)]
    m = dict(macro_f1=f1_score(y_true, pred, average='macro', zero_division=0),
             weighted_f1=f1_score(y_true, pred, average='weighted', zero_division=0),
             mcc=matthews_corrcoef(y_true, pred))
    Y = label_binarize(y_true, classes=list(classes))
    if Y.shape[1] == 1:
        Y = np.hstack([1 - Y, Y])
    try:
        m['auroc'] = roc_auc_score(Y, P, average='macro', multi_class='ovr')
    except ValueError:
        m['auroc'] = np.nan
    try:
        m['auprc'] = average_precision_score(Y, P, average='macro')
    except ValueError:
        m['auprc'] = np.nan
    m['brier'] = float(np.mean(np.sum((P - Y) ** 2, axis=1)))
    m['ece'] = expected_calibration_error(y_true, pred, P.max(axis=1), n_bins=CFG['ece_bins'])
    ben_col = np.where(is_benign(classes))[0]
    assert len(ben_col) == 1, f'benign column not unique in {classes}'
    t_att = ~is_benign(y_true); p_att = P.argmax(1) != ben_col[0]; s_att = 1.0 - P[:, ben_col[0]]
    m['n_benign'] = int((~t_att).sum()); m['n_attack'] = int(t_att.sum())
    m['fp_rate'] = float(p_att[~t_att].mean()) if (~t_att).any() else np.nan
    m['attack_recall'] = float(p_att[t_att].mean()) if t_att.any() else np.nan
    if t_att.any() and (~t_att).any():
        m['bin_auroc'] = roc_auc_score(t_att, s_att); m['bin_auprc'] = average_precision_score(t_att, s_att)
    else:
        m['bin_auroc'] = m['bin_auprc'] = np.nan
    return m

METRICS = ['macro_f1', 'weighted_f1', 'mcc', 'auroc', 'auprc', 'brier', 'ece', 'n_benign', 'n_attack',
           'fp_rate', 'attack_recall', 'bin_auroc', 'bin_auprc']

def done_keys(path, keys):
    if not os.path.exists(path):
        return set()
    df = pd.read_csv(path)
    return set(map(tuple, df[keys].astype(str).to_numpy()))

def append(path, rows, cols):
    df = pd.DataFrame(rows)[cols]
    df.to_csv(path, mode='a', header=not os.path.exists(path), index=False)

## Stage 2 — Fit the nine source models and store their scores

Stratified 80% of the source per seed, exactly as notebook 21, so the models are the same fits. Stores
P(attack) for the in-domain test rows and for every row of both days. Skips a model whose score file
exists.

In [ ]:
def score_path(mname, seed): return f'{SC}/scores_{mname}_seed{seed}.parquet'
for mname in CFG['models']:
    for seed in CFG['seeds']:
        if os.path.exists(score_path(mname, seed)):
            print(f'{mname} seed {seed}: scores exist, skipped'); continue
        tr, te = train_test_split(np.arange(len(yS)), train_size=CFG['source_train_frac'],
                                  stratify=yS, random_state=seed)
        t0 = time.time(); mdl = make_model(mname, seed).fit(XS[tr], yS[tr]); fit_s = time.time() - t0
        classes = np.asarray(mdl.classes_).astype(str); assert list(classes) == FAMS, classes
        ben_col = np.where(is_benign(classes))[0][0]
        t1 = time.time()
        s_in = 1 - mdl.predict_proba(XS[te])[:, ben_col]
        s_T = 1 - mdl.predict_proba(XT)[:, ben_col]
        score_s = time.time() - t1
        pd.DataFrame({'part': np.r_[np.repeat('in_domain', len(te)), np.repeat('target', len(XT))],
                      'row': np.r_[te, np.arange(len(XT))].astype(np.int64),
                      'label': np.r_[yS[te], yT].astype(str),
                      'p_attack': np.r_[s_in, s_T].astype(np.float32),
                      'fit_s': np.float32(fit_s)}).to_parquet(score_path(mname, seed), index=False)
        m = panel(yS[te], two_col_scores(s_in, classes), classes)
        print(f"{mname} seed {seed}: fit {fit_s:.0f}s, scoring {score_s:.0f}s | in-domain MCC {m['mcc']:.3f} "
              f"AUROC {m['bin_auroc']:.3f}")
        del mdl; gc.collect()
print('Stage 2 complete')

## Stage 3 — Evaluations from stored scores

Per model, seed and day:
* `eval = whole`: zero-shot on the whole day.
* `eval = own`: for each labelled sample size p, the unchanged detector, Platt, isotonic and the
  prior-only baseline fitted on the first p% and scored on the remaining rows (notebook 21's layout).
* `eval = common`: the same fits scored on the rows after the first 10% of the day, so every budget and
  method is compared on the same flows; the zero-shot row of this block has `frac = 0`.

Also written: per-attack-type AUROC and alarm rate (`conf23_types.csv`, whole day for zero-shot and the
common part for every method at 10%), and the labelled-sample composition with the detector's AUROC
inside the sample (`conf23_prefix.csv`).

In [ ]:
R_CSV, T_CSV, X_CSV = f'{OUT}/conf23_transfer.csv', f'{OUT}/conf23_types.csv', f'{OUT}/conf23_prefix.csv'
R_KEYS = ['model', 'seed', 'regime', 'eval', 'method', 'frac']
R_COLS = R_KEYS + ['n_eval', 'n_buffer', 'buffer_benign', 'rho_hat', 'platt_a'] + METRICS + ['fit_s']
T_COLS = R_KEYS + ['attack_type', 'n', 'flagged_attack', 'auroc_vs_benign']
X_COLS = ['model', 'seed', 'regime', 'frac', 'n_buffer', 'buffer_benign', 'buffer_attack', 'attack_types',
          'auroc_in_sample', 'auroc_whole_day']
CLASSES = np.asarray(FAMS)

def type_rows(base, nat, yv, P, classes):
    ben_col = np.where(is_benign(classes))[0][0]
    s = 1 - P[:, ben_col]; ben = is_benign(yv); out = []
    for t in sorted(set(nat[~ben])):
        msk = (nat == t) & ~ben
        if msk.sum() < CFG['native_min_rows'] or ben.sum() < CFG['native_min_rows']:
            continue
        lab = np.r_[np.ones(msk.sum()), np.zeros(ben.sum())]
        out.append(dict(**base, attack_type=t, n=int(msk.sum()),
                        flagged_attack=float((P[msk].argmax(1) != ben_col).mean()),
                        auroc_vs_benign=roc_auc_score(lab, np.r_[s[msk], s[ben]])))
    return out

def row(base, n_eval, nb, nbb, rho, a, fit_s, m):
    return dict(**base, n_eval=int(n_eval), n_buffer=int(nb), buffer_benign=int(nbb), rho_hat=rho, platt_a=a,
                fit_s=round(float(fit_s), 1), **m)

done = done_keys(R_CSV, ['model', 'seed'])
pc = CFG['common_frac']
for mname in CFG['models']:
    for seed in CFG['seeds']:
        if (mname, str(seed)) in done:
            continue
        S = pd.read_parquet(score_path(mname, seed)); fit_s = float(S.fit_s.iloc[0])
        sT = S[S.part == 'target'].sort_values('row').p_attack.to_numpy(float)
        assert len(sT) == len(XT)
        rows, trows, xrows = [], [], []
        for r, order in ORD.items():
            s = sT[order]; yv = yT[order]; nat = nT[order]; P = two_col_scores(s, CLASSES)
            base = dict(model=mname, seed=seed, regime=r)
            auc_day = roc_auc_score(~is_benign(yv), s)
            rows.append(row(dict(**base, eval='whole', method='none', frac=0.0), len(order), 0, 0, np.nan, np.nan, fit_s,
                            panel(yv, P, CLASSES)))
            trows += type_rows(dict(**base, eval='whole', method='none', frac=0.0), nat, yv, P, CLASSES)
            bufC, restC = time_ordered_buffer(np.arange(len(order)), pc)
            rows.append(row(dict(**base, eval='common', method='none', frac=0.0), len(restC), 0, 0, np.nan, np.nan, fit_s,
                            panel(yv[restC], P[restC], CLASSES)))
            trows += type_rows(dict(**base, eval='common', method='none', frac=0.0), nat[restC], yv[restC], P[restC], CLASSES)
            for frac in CFG['fracs']:
                buf, rest = time_ordered_buffer(np.arange(len(order)), frac)
                nb = len(buf); nbb = int(is_benign(yv[buf]).sum())
                ben_b = is_benign(yv[buf])
                auc_in = roc_auc_score(~ben_b, s[buf]) if ben_b.any() and (~ben_b).any() else np.nan
                xrows.append(dict(model=mname, seed=seed, regime=r, frac=frac, n_buffer=nb, buffer_benign=nbb,
                                  buffer_attack=nb - nbb,
                                  attack_types=json.dumps({str(k): int(v) for k, v in pd.Series(nat[buf][~ben_b]).value_counts().items()}),
                                  auroc_in_sample=auc_in, auroc_whole_day=auc_day))
                rows.append(row(dict(**base, eval='own', method='none', frac=frac), len(rest), nb, nbb, np.nan, np.nan, fit_s,
                                panel(yv[rest], P[rest], CLASSES)))
                Pp, rho = prior_only(P[buf], yv[buf], len(rest), CLASSES)
                rows.append(row(dict(**base, eval='own', method='prior', frac=frac), len(rest), nb, nbb, rho, np.nan, fit_s,
                                panel(yv[rest], Pp, CLASSES)))
                PpC, _ = prior_only(P[buf], yv[buf], len(restC), CLASSES)
                rows.append(row(dict(**base, eval='common', method='prior', frac=frac), len(restC), nb, nbb, rho, np.nan, fit_s,
                                panel(yv[restC], PpC, CLASSES)))
                for meth in CFG['methods']:
                    if ben_b.all() or (~ben_b).all():
                        for ev, n_ in [('own', len(rest)), ('common', len(restC))]:
                            rows.append(row(dict(**base, eval=ev, method=meth, frac=frac), n_, nb, nbb, rho, np.nan, fit_s,
                                            {k: np.nan for k in METRICS}))
                        continue
                    Pc, cal = calibrate(P[buf], yv[buf], P[rest], CLASSES, meth)
                    a = platt_slope(cal) if meth == 'sigmoid' else np.nan
                    rows.append(row(dict(**base, eval='own', method=meth, frac=frac), len(rest), nb, nbb, rho, a, fit_s,
                                    panel(yv[rest], Pc, CLASSES)))
                    PcC = cal.predict_proba(P[restC])
                    rows.append(row(dict(**base, eval='common', method=meth, frac=frac), len(restC), nb, nbb, rho, a, fit_s,
                                    panel(yv[restC], PcC, CLASSES)))
                    if frac == CFG['fracs'][-1]:
                        trows += type_rows(dict(**base, eval='common', method=meth, frac=frac), nat[restC], yv[restC], PcC, CLASSES)
        append(R_CSV, rows, R_COLS); append(T_CSV, trows, T_COLS); append(X_CSV, xrows, X_COLS)
        df_ = pd.DataFrame(rows)
        for r in ORD:
            z = df_[(df_.regime == r) & (df_['eval'] == 'common') & (df_.method == 'none')].iloc[0]
            c = df_[(df_.regime == r) & (df_['eval'] == 'common') & (df_.method == 'sigmoid') & (df_.frac == 0.01)].iloc[0]
            print(f"{mname:5s} seed {seed} {r}: common part, zero-shot AUROC {z.bin_auroc:.3f} MCC {z.mcc:.3f} | "
                  f"Platt 1% MCC {c.mcc:.3f} ECE {c.ece:.3f} slope {c.platt_a:+.2f}")
        del S, sT; gc.collect()
print('Stage 3 complete')

## Stage 4 — Drift per capture day, Portmap included

As in notebook 21: PSI and KS on every common-core feature between the source and each day, for all,
benign-only and attack-only traffic, on at most 300,000 rows per side. Writes `conf23_drift_summary.csv`.

In [ ]:
rng = np.random.default_rng(0)
def sample(idx, n=CFG['drift_sample']):
    return idx if len(idx) <= n else rng.choice(idx, n, replace=False)
ben_S = is_benign(yS); feat_rows, summ = [], []
for r, rows in REG_ROWS.items():
    ben_T = is_benign(yT[rows])
    for subset, iS, iT in [('all', np.arange(len(yS)), rows),
                           ('benign', np.where(ben_S)[0], rows[ben_T]),
                           ('attack', np.where(~ben_S)[0], rows[~ben_T])]:
        a, b = XS[sample(iS)], XT[sample(iT)]
        ps = [psi(a[:, j], b[:, j]) for j in range(len(CORE))]
        ks = [ks_stat(a[:, j], b[:, j]) for j in range(len(CORE))]
        feat_rows += [dict(regime=r, subset=subset, feature=f, psi=p, ks=k) for f, p, k in zip(CORE, ps, ks)]
        ps = np.asarray(ps); ks = np.asarray(ks)
        summ.append(dict(regime=r, subset=subset, n_features=len(CORE), n_psi_gt_025=int((ps > 0.25).sum()),
                         median_psi=float(np.nanmedian(ps)), median_ks=float(np.nanmedian(ks)),
                         max_ks=float(np.nanmax(ks))))
pd.DataFrame(feat_rows).to_csv(f'{OUT}/conf23_drift_features.csv', index=False)
drift_summary = pd.DataFrame(summ)
drift_summary.to_csv(f'{OUT}/conf23_drift_summary.csv', index=False)
print(drift_summary.round(3).to_string(index=False))

## Stage 5 — Summaries

Means and standard deviations over seeds. Writes `conf23_transfer_summary.csv`, `conf23_types_summary.csv`
and `conf23_prefix_summary.csv`, and prints the headline numbers on the common evaluation part.

In [ ]:
def agg(df, keys, cols):
    g = df.groupby(keys)[cols].agg(['mean', 'std'])
    g.columns = [f'{a}_{b}' for a, b in g.columns]
    return g.reset_index()

R = pd.read_csv(R_CSV).drop_duplicates(R_KEYS, keep='last')
T = pd.read_csv(T_CSV).drop_duplicates(R_KEYS + ['attack_type'], keep='last')
X = pd.read_csv(X_CSV).drop_duplicates(['model', 'seed', 'regime', 'frac'], keep='last')
cols = ['macro_f1', 'mcc', 'attack_recall', 'fp_rate', 'bin_auroc', 'bin_auprc', 'ece', 'platt_a', 'rho_hat']
SR = agg(R, ['regime', 'eval', 'method', 'frac', 'model'], cols); SR.to_csv(f'{OUT}/conf23_transfer_summary.csv', index=False)
ST = agg(T, ['regime', 'eval', 'method', 'frac', 'attack_type', 'model'], ['auroc_vs_benign', 'flagged_attack', 'n'])
ST.to_csv(f'{OUT}/conf23_types_summary.csv', index=False)
SX = agg(X, ['regime', 'frac', 'model'], ['auroc_in_sample', 'auroc_whole_day'])
SX = SX.merge(X[X.seed == CFG['seeds'][0]][['regime', 'frac', 'model', 'n_buffer', 'buffer_benign', 'buffer_attack', 'attack_types']],
              on=['regime', 'frac', 'model']); SX.to_csv(f'{OUT}/conf23_prefix_summary.csv', index=False)

print('Zero-shot, whole day:')
print(SR[(SR['eval'] == 'whole')][['regime', 'model', 'bin_auroc_mean', 'mcc_mean', 'mcc_std', 'ece_mean', 'attack_recall_mean', 'fp_rate_mean']].round(3).to_string(index=False))
print('\nCommon evaluation part (rows after the first 10% of each day):')
show = SR[(SR['eval'] == 'common')]
print(show[['regime', 'model', 'method', 'frac', 'bin_auroc_mean', 'mcc_mean', 'mcc_std', 'ece_mean', 'attack_recall_mean', 'fp_rate_mean', 'platt_a_mean']].round(3).to_string(index=False))
print('\nLabelled samples (seed 0 counts; AUROC mean over seeds):')
print(SX[['regime', 'frac', 'model', 'n_buffer', 'buffer_benign', 'buffer_attack', 'auroc_in_sample_mean', 'auroc_whole_day_mean', 'attack_types']].round(3).to_string(index=False))
print('\nNegative Platt slopes (order reversed):', int((R.platt_a < 0).sum()), 'of', int(R.platt_a.notna().sum()), 'fits')
print(R[R.platt_a < 0].groupby(['model', 'regime']).size().to_dict())

## Stage 6 — Paper figures and table values

Figure 3 (ranking before against decisions after recalibration, and ECE against decisions) on the
common evaluation part; Figure 4 (per-attack-type AUROC, whole day, zero-shot); Table II values; the
labelled-sample table; and the numbers quoted in the text. 600 dpi PNG and vector PDF, paper names on
every axis. Written to `results/conference/paper/`.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import spearmanr
COL = {'rf': '#1f5f99', 'lgbm': '#d9792b', 'mlp': '#3c8d3c'}
plt.rcParams.update({'font.family': 'serif', 'font.size': 7, 'axes.titlesize': 7.5, 'axes.labelsize': 7,
                     'legend.fontsize': 6.5, 'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5,
                     'axes.spines.top': False, 'axes.spines.right': False, 'pdf.fonttype': 42,
                     'savefig.bbox': 'tight', 'savefig.pad_inches': 0.02})
def save(fig, name):
    fig.savefig(f'{PAP}/{name}.pdf'); fig.savefig(f'{PAP}/{name}.png', dpi=CFG['dpi']); plt.close(fig); print('saved', name)

# --- Figure 3: common evaluation part ---
com = R[R['eval'] == 'common']
z = com[(com.method == 'none')][['model', 'seed', 'regime', 'bin_auroc']].rename(columns={'bin_auroc': 'auroc_zero'})
c = com[com.method.isin(CFG['methods'])]
D = c.merge(z, on=['model', 'seed', 'regime']); D = D[D.mcc.notna()]
rho_a = spearmanr(D.auroc_zero, D.mcc).statistic; rho_e = spearmanr(D.ece, D.mcc).statistic
low = D[D.ece < 0.1]
fig, axes = plt.subplots(1, 2, figsize=(7.16, 1.85)); MK = {'regime_1': 'o', 'regime_2': 's'}
for ax, xcol in [(axes[0], 'auroc_zero'), (axes[1], 'ece')]:
    for m in CFG['models']:
        for r in ORD:
            for meth, filled in [('sigmoid', True), ('isotonic', False)]:
                s_ = D[(D.model == m) & (D.regime == r) & (D.method == meth)]
                ax.scatter(s_[xcol], s_.mcc, s=13, marker=MK[r], c=COL[m] if filled else 'white', edgecolors=COL[m],
                           linewidths=0.7, alpha=0.9, zorder=3)
    ax.axhline(0, color='grey', lw=0.5, ls=':'); ax.set_ylim(-0.08, 0.82); ax.set_ylabel('MCC after recalibration')
axes[0].axvline(0.5, color='grey', lw=0.5, ls=':'); axes[0].set_xlim(0.38, 0.97)
axes[0].set_xlabel('AUROC on the same flows before recalibration')
axes[0].set_title(f'(a) Ranking before vs decisions after (Spearman {rho_a:.2f})', loc='left')
axes[1].axvspan(0, 0.1, color='#e5f5e0', zorder=0); axes[1].text(0.05, 0.78, 'ECE < 0.1', ha='center', fontsize=6)
axes[1].set_xlabel('ECE after recalibration (lower = better calibrated)'); axes[1].set_xlim(0, max(0.62, float(D.ece.max()) + 0.02))
axes[1].set_title(f'(b) Calibration error vs decisions after (Spearman {rho_e:.2f})', loc='left')
pr = com[com.method == 'prior']
axes[1].scatter(pr.ece, pr.mcc, s=18, marker='x', c='black', linewidths=0.8, zorder=4)
hd = [Line2D([], [], ls='none', marker='o', color=COL[m], label=MODEL_NAME[m], ms=4) for m in CFG['models']]
hd += [Line2D([], [], ls='none', marker='o', mfc='grey', mec='grey', label='Day 1', ms=4),
       Line2D([], [], ls='none', marker='s', mfc='grey', mec='grey', label='Day 2', ms=4),
       Line2D([], [], ls='none', marker='o', mfc='grey', mec='grey', label='Platt (filled)', ms=4),
       Line2D([], [], ls='none', marker='o', mfc='white', mec='grey', label='Isotonic (open)', ms=4),
       Line2D([], [], ls='none', marker='x', color='black', label='Prior only', ms=4)]
fig.legend(handles=hd, frameon=False, loc='upper center', ncol=8, bbox_to_anchor=(0.5, 1.07), columnspacing=0.9, handletextpad=0.2)
fig.tight_layout(w_pad=1.5); save(fig, 'fig3_transfer'); D.to_csv(f'{PAP}/fig3_values.csv', index=False)

# --- Figure 4: per-type AUROC, whole day, zero-shot ---
zt = T[(T['eval'] == 'whole') & (T.method == 'none')]
H = zt.groupby(['regime', 'attack_type', 'model'])['auroc_vs_benign'].mean().unstack('model')[CFG['models']]
SHORT = lambda o: o.replace('DrDoS_', '').replace('UDPLag', 'UDP-lag')
NN = zt[(zt.model == 'rf') & (zt.seed == CFG['seeds'][0])].set_index(['regime', 'attack_type'])['n']
ORDER = {}
for r in ORD:
    idx = list(H.loc[r].index)
    tail = [x for x in ['Syn', 'UDP-lag', 'UDPLag', 'WebDDoS'] if x in idx]
    ORDER[r] = sorted([x for x in idx if x not in tail], key=SHORT) + tail
n1, n2 = len(ORDER['regime_1']), len(ORDER['regime_2'])
fig, axes = plt.subplots(1, 2, figsize=(7.16, 1.38), gridspec_kw={'width_ratios': [n1, n2]})
for ax_, r in zip(axes, ['regime_1', 'regime_2']):
    o = ORDER[r]; M = H.loc[r].loc[o].T.to_numpy()
    im = ax_.imshow(M, cmap='RdBu', vmin=0.0, vmax=1.0, aspect='auto')
    ax_.set_yticks(range(3)); ax_.set_yticklabels([MODEL_NAME[m] for m in CFG['models']] if r == 'regime_1' else [])
    ax_.set_xticks(range(len(o)))
    ax_.set_xticklabels([f'{SHORT(x)} ({NN.loc[(r, x)] / 1000:.0f}k)' if NN.loc[(r, x)] >= 1000 else f'{SHORT(x)} ({NN.loc[(r, x)]})'
                         for x in o], fontsize=5.8, rotation=35, ha='right', rotation_mode='anchor')
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax_.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center', fontsize=5.8, color='white' if abs(M[i, j] - 0.5) > 0.3 else 'black')
    ax_.set_title(f'({"a" if r == "regime_1" else "b"}) {DAY_NAME[r]}', loc='left'); ax_.tick_params(length=0)
    for sp in ax_.spines.values(): sp.set_visible(False)
cb = fig.colorbar(im, ax=axes, fraction=0.02, pad=0.01); cb.ax.tick_params(labelsize=6); cb.set_label('AUROC against benign', fontsize=6.2)
save(fig, 'fig4_attack_types'); H.round(3).to_csv(f'{PAP}/fig4_values.csv')

# --- Table II values and quoted numbers ---
t2 = SR[['regime', 'eval', 'method', 'frac', 'model', 'bin_auroc_mean', 'mcc_mean', 'mcc_std', 'ece_mean', 'attack_recall_mean',
         'fp_rate_mean', 'platt_a_mean', 'rho_hat_mean']]
t2.round(3).to_csv(f'{PAP}/table2_transfer.csv', index=False)
SX.round(3).to_csv(f'{PAP}/table_prefix.csv', index=False)
nums = {}
w = R[(R['eval'] == 'whole') & (R.model == 'rf') & (R.seed == CFG['seeds'][0])].set_index('regime')
for r in ORD:
    nums[f'{r}_flows'] = int(w.loc[r, 'n_eval']); nums[f'{r}_benign'] = int(w.loc[r, 'n_benign'])
    nums[f'{r}_benign_share'] = w.loc[r, 'n_benign'] / w.loc[r, 'n_eval']
for r in ORD:
    n_ = NN.loc[r]; tail = [x for x in ['Syn', 'UDP-lag', 'UDPLag', 'WebDDoS'] if x in n_.index]
    nums[f'{r}_tail_types_share_of_attacks'] = n_[tail].sum() / n_.sum()
    for x in n_.index: nums[f'{r}_n_{x}'] = int(n_[x])
nums['spearman_auroc_before_mcc_after_common'] = rho_a; nums['spearman_ece_after_mcc_after_common'] = rho_e
nums['n_runs'] = len(D); nums['n_ece_below_0.1'] = len(low); nums['n_ece_below_0.1_and_mcc_below_0.1'] = int((low.mcc < 0.1).sum())
hi = D[D.mcc > 0.55]; nums['runs_mcc_above_0.55'] = json.dumps({f'{k[0]}_{k[1]}': int(v) for k, v in hi.groupby(['model', 'regime']).size().items()})
nums['negative_platt_slopes'] = json.dumps({f'{k[0]}_{k[1]}': int(v) for k, v in R[R.platt_a < 0].groupby(['model', 'regime']).size().items()})
pri = SR[(SR['eval'] == 'common') & (SR.method == 'prior')]
for _, x in pri.iterrows():
    if x.model == 'rf': nums[f'prior_only_common_{x.regime}_frac{x.frac:.2f}_ece'] = x.ece_mean; nums[f'prior_only_common_{x.regime}_frac{x.frac:.2f}_rho'] = x.rho_hat_mean
pd.Series(nums, name='value').to_csv(f'{PAP}/paper_numbers.csv'); print(pd.Series(nums).to_string())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
os.chdir('/content/drive/MyDrive/drift-conference')

r = subprocess.run(["python", "tools/commit_cell.py",
  "23: transfer with Portmap kept, matched evaluation, prior-only baseline, labelled-sample composition"],
  capture_output=True, text=True)
print(r.stdout)
print(r.stderr)